In [1]:
import sys
import time
import random
import json
import textwrap
from pymongo import MongoClient
from IPython.display import clear_output

# Helper function to wrap text without losing existing paragraph breaks
def wrap_text(text, width=100):
    if not text or not isinstance(text, str):
        return str(text)
    return "\n".join([textwrap.fill(line, width=width) for line in text.splitlines()])

with open("config.json") as f:
    config = json.load(f)

# --- Database Connection ---
client = MongoClient(config["mongo_uri"])
db = client["consensus"]
collection = db["python_source_map_comments"]           
collection_source_map = db["python_source_map"]  

sampled_docs = []
seen_ids = []

# --- 1. Identify and Exclude Original Posts ---
print("Mapping original posts to exclude them from the coding queue...")
first_post_pipeline = [
    {"$match": {"created_at": {"$exists": True, "$ne": None}}}, 
    {"$sort": {"Episode #": 1, "created_at": 1}}, 
    {"$group": {
        "_id": "$Episode #",
        "first_comment_id": {"$first": "$_id"}
    }}
]
first_post_docs = list(collection.aggregate(first_post_pipeline))
excluded_first_posts = [doc["first_comment_id"] for doc in first_post_docs if "first_comment_id" in doc]

# Add them to seen_ids so they are never sampled
seen_ids.extend(excluded_first_posts)
print(f"Excluded {len(excluded_first_posts)} original posts.\n")

# --- 2. Fetch Specific Episodes by Integer Range ---
episode_ranges = [
    {"$gte": 52, "$lte": 58},
    {"$gte": 59, "$lte": 86}
]

for ep_range in episode_ranges:
    doc = collection.find_one({
        "Episode #": ep_range,
        "manual_valence": {"$exists": False},
        "_id": {"$nin": seen_ids} # Prevent picking an original post or duplicate
    })
    if doc:
        sampled_docs.append(doc)
        seen_ids.append(doc["_id"])

# --- 3. Stratified Sampling (30 total: 10 per valence) ---
sample_sizes = {-1: 10, 0: 10, 1: 10}
stratified_docs = []

for val, limit in sample_sizes.items():
    pipeline = [
        {"$match": {
            "valence": val,          
            "manual_valence": {"$exists": False},
            "_id": {"$nin": seen_ids} # Prevent duplicates & original posts
        }},
        {"$sample": {"size": limit}}
    ]
    docs = list(collection.aggregate(pipeline))
    stratified_docs.extend(docs)

# Shuffle only the stratified portion so the specific episodes stay at the front
random.shuffle(stratified_docs)
sampled_docs.extend(stratified_docs)

print(f"Loaded {len(sampled_docs)} uncoded replies for review.\n")
time.sleep(1.5) 

# --- Manual Coding Loop ---
for i, doc in enumerate(sampled_docs, 1):
    
    episode_id = doc.get('Episode #')
    first_comment_text = "No context available."
    choice_value = "N/A"
    
    if episode_id is not None:
        # 1. Fetch Context (Using created_at)
        first_comment = collection.find_one(
            {
                "Episode #": episode_id,
                "created_at": {"$exists": True, "$ne": None} 
            },
            sort=[("created_at", 1)]
        )
        
        if first_comment:
            first_comment_text = first_comment.get('text', 'No text found')
            
        # 2. Fetch Choice from python_source_map
        source_map_doc = collection_source_map.find_one({"Episode #": episode_id})
        if source_map_doc:
            choice_value = source_map_doc.get('Choice', 'N/A')

    # Apply text wrapping
    wrapped_context = wrap_text(first_comment_text)
    wrapped_current_text = wrap_text(doc.get('text', 'No text found'))

    # 3. Render UI
    clear_output(wait=False) 
    
    print(f"--- Comment {i} of {len(sampled_docs)} ---")
    print(f"Target: {doc.get('valence_meta', {}).get('target', 'N/A')} | Episode #: {episode_id} | Choice: {choice_value}")
    print("-" * 100)
    print("CONTEXT (First Comment of Episode):")
    print(f"\n{wrapped_context}\n")
    print("-" * 100)
    print("=" * 100)
    print(f"TEXT TO CODE (Reply to Context):\n\n{wrapped_current_text}\n")
    print("=" * 100)
    
    sys.stdout.flush()
    time.sleep(0.05) 
    
    # 4. Valence Input
    while True:
        val_input = input("1. Enter valence (-1, 0, 1) [or 'q' to quit, 'skip' to skip]: ").strip().lower()
        
        if val_input in ['q', 'skip', '-1', '0', '1']:
            break
        print("Invalid input. Please type -1, 0, 1, 'skip', or 'q'.")
            
    if val_input == 'q':
        print("Session ended early.")
        break
    elif val_input == 'skip':
        print("Skipped.")
        continue

    # 5. Update document in MongoDB
    update_data = {"manual_valence": int(val_input)}

    collection.update_one(
        {"_id": doc["_id"]},
        {"$set": update_data}
    )

print("\nCoding session complete!")

--- Comment 32 of 32 ---
Target: 634 | Episode #: 89 | Choice: Accept PEP 634 as-is
----------------------------------------------------------------------------------------------------
CONTEXT (First Comment of Episode):

The SC is trying to see whether there is clear consensus on pattern matching in general and PEP 634
specifically. Please vote below to express your current feeling on this topic. To be clear, this is
not a binding vote, but instead to see if any consensus exists among the core team. This poll will
close on Monday, Nov 23. Accept PEP 634 as-is Accept PEP 634 + 640 Accept PEP 634 + 642 I want
pattern matching, but not as defined in those PEPs I don’t want pattern matching 0 voters

----------------------------------------------------------------------------------------------------
TEXT TO CODE (Reply to Context):

I voted for PEP 634 as-is. My other consideration would have been PEP 634 + PEP 640. But, I saw PEP
640 as a standalone proposal in itself. The use of _ for c

In [3]:
import json
from pymongo import MongoClient
from bson import json_util

# --- Setup Connection ---
with open("config.json") as f:
    config = json.load(f)

client = MongoClient(config["mongo_uri"])
db = client["consensus"]
collection = db["python_source_map_comments"]           

print("Fetching documents from MongoDB...")

# Option A: Export ALL comments
# docs = list(collection.find())

# Option B: Export ONLY the comments you have manually coded
docs = list(collection.find({"manual_valence": {"$exists": True}}))

if not docs:
    print("No documents found to export.")
else:
    filename = "coded_comments_export.json"
    
    # Use bson.json_util to properly handle MongoDB-specific types like ObjectId and Date
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(docs, f, default=json_util.default, indent=4)
        
    print(f"Successfully exported {len(docs)} records to {filename}")

Fetching documents from MongoDB...
Successfully exported 40 records to coded_comments_export.json
